# SC4001 – Object hallucination in Qwen2-VL-2B (Colab runner)

This notebook only **runs** code. All code lives in the GitHub repo and is edited on your laptop (Claude Code), then pushed.
Every output goes to Google Drive, so a disconnect loses nothing: just re-run the same cell and it resumes.

**Before the first run:** Runtime → Change runtime type → **T4 GPU**.

Run cells 1–5 at the start of every session. Cells marked *once* are only needed once for the whole team.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Google Drive

`DRIVE_ROOT` is the shared project folder. The owner creates `MyDrive/sc4001` and shares it with teammates (Editor).
Teammates: open *Shared with me*, right-click `sc4001` → **Organize → Add shortcut → My Drive**. Then the same path works for everyone.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/sc4001'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.environ['SC4001_ROOT'] = DRIVE_ROOT   # read by src/halluc/config.py
print('Project folder:', DRIVE_ROOT, os.listdir(DRIVE_ROOT))

## 3. Get the latest code from GitHub

Set `REPO` to your repo. If the repo is **private**, add a Colab secret (🔑 icon in the left sidebar) named `GITHUB_TOKEN`
holding a GitHub fine-grained token with *read* access to this repo, and switch on *Notebook access*.

In [ ]:
REPO = 'Kxtnny/sc4001-vlm-hallucination'
BRANCH = 'main'
REPO_DIR = '/content/repo'

try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'

import subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)
%cd {REPO_DIR}
!git log --oneline -3

## 4. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
import transformers, torch; print('transformers', transformers.__version__, '| torch', torch.__version__)

## 5. Unit tests (CPU, ~10 s)
Checks answer parsing, metrics, subsets and the image-token / attention helpers on a tiny random Qwen2-VL.

In [ ]:
!python -m pytest -q tests

## 6. Prepare data (*once for the team*, ~5–10 min)

Downloads the POPE files, COCO 2014 annotations and ~1,200 COCO val2014 images to Drive, and builds the frozen splits:
`pope` (500 test images), `chair_test` (500 test images), `dev` (200 images for tuning). Safe to re-run: it skips what exists and never rebuilds the splits.

In [ ]:
!python scripts/prepare_data.py

## 7. E0 – sanity checks

Prints 10 POPE answers to read by hand, checks image-token positions, that attention rows sum to 1, that a present object gets "yes",
and times one question. If the float16 check fails (NaN/inf), re-run with `--dtype float32` and use float32 for every run.

In [ ]:
!python scripts/sanity_check.py

## 8. Timing run: 50 questions
The full runs below reuse these answers (same fixed subset), so nothing is wasted.

In [ ]:
!python scripts/eval_pope.py --split random --limit 50

## 9. POPE on a 300-question subset per split (plan: Tue 3 – Wed 4)

In [ ]:
!python scripts/eval_pope.py --split all --limit 300

## 10. E1 – full POPE baseline (3 × 3,000 questions)

Long run. If Colab disconnects, run this cell again; it resumes. You can also run one split per teammate account
(`--split random`, `--split popular`, `--split adversarial`) since they all write to the shared Drive folder.

In [ ]:
!python scripts/eval_pope.py --split all

## 11. Results table (from saved JSON, no GPU needed)

In [ ]:
import json, glob, pandas as pd
rows = [json.load(open(p)) for p in sorted(glob.glob(f'{DRIVE_ROOT}/results/pope/*/*_metrics.json'))]
df = pd.DataFrame(rows)
if len(df):
    cols = ['tag', 'split', 'n', 'complete', 'accuracy', 'precision', 'recall', 'f1', 'yes_ratio', 'n_parse_fallback']
    display(df[cols].round(3))

## 12. Log the run

Add a line to `RUN_LOG.md` in the repo (on your laptop, then push): date, who, command, tag, git commit (in `run_meta.json`), and the result file.
The report may only describe runs that appear in this log.